# Notebook 2: Training
Loads processed fitness data, trains the NeuMF model, and records Hit Rate@3 and NDCG@3.

## Data
Loading preprocessed fitness CSVs with negative samples already generated in Notebook 1.

In [8]:
import sys
import pandas as pd
import numpy as np
import torch

sys.path.insert(0, 'src')

train_data = pd.read_csv('src/data/train.csv')
val_data = pd.read_csv('src/data/val.csv')
test_data = pd.read_csv('src/data/test.csv')

print(f"Train: {len(train_data)}")
print(f"Val:   {len(val_data)}")
print(f"Test:  {len(test_data)}")

Train: 3112
Val:   388
Test:  392


## Model Configuration
NeuMF with 8-dimensional embeddings for both GMF and MLP pathways.
973 users, 4 workout types (Yoga, HIIT, Cardio, Strength).

In [9]:
from neumf import NeuMFEngine

config = {
    'alias': 'neumf_fitness',
    'num_users': 973,
    'num_items': 4,
    'latent_dim_mf': 8,
    'latent_dim_mlp': 8,
    'layers': [16, 64, 32, 16, 8],
    'l2_regularization': 0.0000001,
    'weight_init_gaussian': True,
    'use_cuda': False,
    'use_bachify_eval': False,
    'device_id': 0,
    'pretrain': False,
    'pretrain_mf': '',
    'pretrain_mlp': '',
    'model_dir': 'checkpoints/{}_Epoch{}_HR{:.4f}_NDCG{:.4f}.model',
    'optimizer': 'adam',
    'adam_lr': 1e-3,
    'num_epoch': 10,
    'batch_size': 256,
    'num_negative': 4,
}

engine = NeuMFEngine(config)
print("Model ready!")
print(engine.model)

Embedding(973, 8)
Embedding(4, 8)
Embedding(973, 8)
Embedding(4, 8)
Linear(in_features=16, out_features=64, bias=True)
Linear(in_features=64, out_features=32, bias=True)
Linear(in_features=32, out_features=16, bias=True)
Linear(in_features=16, out_features=8, bias=True)
Linear(in_features=16, out_features=1, bias=True)
NeuMF(
  (embedding_user_mlp): Embedding(973, 8)
  (embedding_item_mlp): Embedding(4, 8)
  (embedding_user_mf): Embedding(973, 8)
  (embedding_item_mf): Embedding(4, 8)
  (fc_layers): ModuleList(
    (0): Linear(in_features=16, out_features=64, bias=True)
    (1): Linear(in_features=64, out_features=32, bias=True)
    (2): Linear(in_features=32, out_features=16, bias=True)
    (3): Linear(in_features=16, out_features=8, bias=True)
  )
  (affine_output): Linear(in_features=16, out_features=1, bias=True)
  (logistic): Sigmoid()
)
Model ready!
NeuMF(
  (embedding_user_mlp): Embedding(973, 8)
  (embedding_item_mlp): Embedding(4, 8)
  (embedding_user_mf): Embedding(973, 8)
  

## DataLoader
Converting CSVs to PyTorch tensors. Shuffling training data each epoch.

In [10]:
from torch.utils.data import DataLoader, TensorDataset

def make_loader(data, batch_size=256):
    users = torch.LongTensor(data['userId'].values.copy())
    items = torch.LongTensor(data['itemId'].values.copy())
    ratings = torch.FloatTensor(data['rating'].values.copy())
    dataset = TensorDataset(users, items, ratings)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)

train_loader = make_loader(train_data)
val_loader = make_loader(val_data)
test_loader = make_loader(test_data)

print("DataLoaders ready!")

DataLoaders ready!


## Popularity Baseline
Always recommend the most popular workout type.
NeuMF needs to outperform this to justify the added complexity.

In [11]:
# Popularity baseline
most_popular_item = train_data[train_data['rating'] == 1]['itemId'].value_counts().index[0]

correct = 0
total = 0
for user_id, group in val_data.groupby('userId'):
    positive = group[group['rating'] == 1]
    if len(positive) == 0:
        continue
    total += 1
    if positive['itemId'].values[0] == most_popular_item:
        correct += 1

baseline_hr = correct / total
print(f"Most popular item: {most_popular_item}")
print(f"Popularity Baseline HR@1: {baseline_hr:.4f}")
print(f"NeuMF HR@3 needs to beat this benchmark")

Most popular item: 3
Popularity Baseline HR@1: 0.2045
NeuMF HR@3 needs to beat this benchmark


## Training
Custom training loop using BCELoss. Evaluating HR@3 and NDCG@3 on validation set after each epoch.

In [12]:
import math
import os

os.makedirs('checkpoints', exist_ok=True)

def evaluate(engine, data):
    engine.model.eval()
    hit_count = 0
    ndcg_sum = 0
    total = 0

    # Group by user
    for user_id, group in data.groupby('userId'):
        users = torch.LongTensor(group['userId'].values.copy())
        items = torch.LongTensor(group['itemId'].values.copy())
        ratings = group['rating'].values.copy()

        with torch.no_grad():
            preds = engine.model(users, items).squeeze().numpy()

        # Rank items by predicted score
        ranked_indices = np.argsort(preds)[::-1]
        top_k = ranked_indices[:3]

        # Find positive item index
        positive_indices = np.where(ratings == 1)[0]
        if len(positive_indices) == 0:
            continue

        pos_idx = positive_indices[0]
        total += 1

        # Hit Rate@3
        if pos_idx in top_k:
            hit_count += 1
            # NDCG@3
            rank = np.where(top_k == pos_idx)[0][0] + 1
            ndcg_sum += 1 / math.log2(rank + 1)

    hr = hit_count / total if total > 0 else 0
    ndcg = ndcg_sum / total if total > 0 else 0
    return hr, ndcg

results = []
for epoch in range(config['num_epoch']):
    engine.model.train()
    for batch in train_loader:
        users_b, items_b, ratings_b = batch
        engine.opt.zero_grad()
        preds = engine.model(users_b, items_b).squeeze()
        loss = engine.crit(preds, ratings_b)
        loss.backward()
        engine.opt.step()

    hr, ndcg = evaluate(engine, val_data)
    results.append({'epoch': epoch+1, 'HR@3': hr, 'NDCG@3': ndcg})
    print(f"Epoch {epoch+1:02d} — HR@3: {hr:.4f}, NDCG@3: {ndcg:.4f}")

print("\nTraining complete!")

Epoch 01 — HR@3: 0.7727, NDCG@3: 0.5381
Epoch 02 — HR@3: 0.7500, NDCG@3: 0.5411
Epoch 03 — HR@3: 0.8182, NDCG@3: 0.5901
Epoch 04 — HR@3: 0.7955, NDCG@3: 0.5950
Epoch 05 — HR@3: 0.7955, NDCG@3: 0.5950
Epoch 06 — HR@3: 0.7955, NDCG@3: 0.5950
Epoch 07 — HR@3: 0.7955, NDCG@3: 0.5950
Epoch 08 — HR@3: 0.7955, NDCG@3: 0.5446
Epoch 09 — HR@3: 0.7500, NDCG@3: 0.5219
Epoch 10 — HR@3: 0.7500, NDCG@3: 0.5041

Training complete!


## Results
Final test set evaluation using HR@3 and NDCG@3.
With only 4 items, @3 is a stricter and more meaningful metric than @10.
NeuMF is compared against a popularity baseline to justify model complexity.

In [13]:
# Evaluate on test set
test_hr, test_ndcg = evaluate(engine, test_data)
print(f"Test HR@3:   {test_hr:.4f}")
print(f"Test NDCG@3: {test_ndcg:.4f}")

# Save results
results_df = pd.DataFrame(results)
results_df.to_csv('src/data/training_results.csv', index=False)
print("\nResults saved to src/data/training_results.csv")

Test HR@3:   0.7692
Test NDCG@3: 0.5302

Results saved to src/data/training_results.csv


In [14]:
# Save model checkpoint
import torch
os.makedirs('checkpoints', exist_ok=True)
torch.save(engine.model.state_dict(), 'checkpoints/neumf_fitness_final.pt')
print("Model saved to checkpoints/neumf_fitness_final.pt")

Model saved to checkpoints/neumf_fitness_final.pt
